# Set Up Notebook

In [ ]:
##########
# IMPORT #
##########

# Data processing and math
import pandas as pd
import numpy as np

# Statistics
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display handling
import warnings


#################
# CONFIGURATION #
#################

# Suppress warnings
warnings.filterwarnings('ignore')

# Set global plotting style
sns.set_theme(style = "whitegrid")


##########
# LABELS #
##########
labels_measure = {
    'measure_action': 'Responsibility for Action',
    'measure_blame':  'Blame / Praise'
}


#################
# VISUALIZATION #
#################
palette_main = {
    "Good":      "#0072B2", "Bad":         "#D55E00",
    "Reflected": "#009E73", "Unreflected": "#CC79A7",
    "Activist":  "#56B4E9", "Bigot":       "#E69F00",
    "Help":      "#F0E442", "Harm":        "#882255"
}

# Transform Data

In [ ]:
# Load data
df = pd.read_csv('blame_praise_self_pilot_2.csv')

# Define factors
def get_factors(n):
    idx = int(n) - 1
    upbringing = "Good" if (idx >> 3) & 1 else "Bad"
    reflection = "Reflected" if (idx >> 2) & 1 else "Unreflected"
    belief = "Bigot" if (idx >> 1) & 1 else "Activist"
    valence = "Harm" if idx & 1 else "Help"
    return upbringing, reflection, belief, valence

# Reshape wide to long
indices = [5, 6, 7, 8, 13, 14, 15, 16]
long_data = []

for _, row in df.iterrows():
    p_id = row['ID']
    
    for i in indices:
        col_action = f'homophobia.{i}-pAction'
        
        blame_cols = [c for c in df.columns if f'blame{i}' in c.replace('.', '')]
        col_blame = blame_cols[0] if blame_cols else None
        
        if col_action in df.columns and pd.notna(row[col_action]):
            up, refl, bel, val = get_factors(i)
            
            score_action = pd.to_numeric(row[col_action], errors = 'coerce') - 7
            
            score_blame = np.nan
            if col_blame and pd.notna(row[col_blame]):
                score_blame = pd.to_numeric(row[col_blame], errors = 'coerce') - 7
            
            long_data.append({
                'ID': p_id,
                'Upbringing': up,
                'Reflection': refl,
                'Belief': bel,
                'Action': val,
                'measure_action': score_action,
                'measure_blame': score_blame
            })

df_long = pd.DataFrame(long_data)
print(f"Transformation complete ({len(df_long)} Observations).")

# Calculate Descriptive Statistics

In [ ]:
# Define group factors and dependent variables
group_factors = ['Upbringing', 'Belief', 'Action']
dependent_vars = ['measure_action', 'measure_blame']

# Calculate descriptive statistics
descriptive_stats = df_long.groupby(group_factors)[dependent_vars].agg(['mean', 'std', 'count']).round(3)

# Display results
display(descriptive_stats)

# Perform ANOVAs

In [ ]:
for dv in ['measure_action', 'measure_blame']:
    print(f"\nANOVA ({labels_measure.get(dv, dv)})")
    
    # Define formula and drop lines with NaN
    formula = f"{dv} ~ C(Upbringing) * C(Belief) * C(Action)"
    df_anova = df_long.dropna(subset=[dv])    
    model = ols(formula, data = df_anova).fit()
    
    # Run ANOVA
    aov_table = sm.stats.anova_lm(model, typ = 2)
    
    # Calculate effect sizes
    aov_table['partial_eta_sq'] = aov_table['sum_sq'] / (aov_table['sum_sq'] + aov_table.loc['Residual', 'sum_sq'])
    
    # Display results
    display(aov_table.round(3))

# Generate Bar Plots

In [ ]:
for measure in ['measure_action', 'measure_blame']:
    current_label = labels_measure.get(measure, measure)
    
    # Create graphs
    g = sns.catplot(data = df_long, x = "Upbringing", y = measure,
                    hue = "Belief", col = "Action", kind = "bar",
                    palette = palette_main, capsize = .05, height = 5)
    
    # Set axis labels and titles
    g.set_axis_labels("Upbringing", f"Mean Score (–6 to +6)")
    g.set_titles("{col_name}")
    
    # Add horizontal zero lines
    for ax in g.axes.flat:
        ax.axhline(0, color = 'black', lw = 1)
        ax.set_ylim(-6, 6)
    
    # Add main title
    plt.subplots_adjust(top = 0.85)
    g.fig.suptitle(current_label, fontsize = 16)
    
    # Export graphs
    filename = f'blame_praise_self_pilot_2_bar_plot_{measure}.png'
    g.savefig(filename, dpi = 300, bbox_inches = 'tight')
    plt.show()
    print(f"Graph saved as '{filename}'.")